In [6]:
# verify imported data matches numbers
import pandas as pd

df = pd.read_pickle("../data/flagged.pkl") # load flagged data into a table

print(df.shape)
print(df.columns.tolist())

has_issue = df["issue_count"] >= 1
print("Numebr of rows without issues:", (len(df)-has_issue.sum()))
print("Percentage of rows with issue(s):", ((has_issue.sum()/len(df))* 100).round(2))

(1067371, 17)
['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'price', 'customer_id', 'country', 'is_duplicate', 'is_cancellation', 'missing_customer', 'missing_description', 'bad_quantity', 'bad_price', 'non_product_code', 'issue_count', 'line_total']
Numebr of rows without issues: 776594
Percentage of rows with issue(s): 27.24


In [7]:
# Order	Rule	        Destination	     Reason
# 1	Duplicate	        Reject	        A copy of a row already counted
# 2	Non-product code    Reject	        Fees and adjustments, not merchandise, even when cancelled
# 3	Cancellation	    Cancellations	Real product returns, tracked separately
# 4	Bad quantity	    Reject	        Stock write-offs, not sales
# 5	Bad price	        Reject	        No money changed hands
# —	Everything else	    Sales	        Includes missing customer and missing description

# turn these rules into column "outcome"

import numpy as np # used to checl conditions

conditions = [
    df["is_duplicate"],
    df["non_product_code"],
    df["is_cancellation"],
    df["bad_quantity"],
    df["bad_price"],
]
outcomes = [
    "rejected: duplicate",
    "rejected: non product code",
    "cancellation",
    "rejected: stock adjustement",
    "rejected: bad price",
]

df["outcome"] = np.select(conditions, outcomes, default="sales")
df["outcome"].value_counts() # shows how many rows got each outcome

print(df["outcome"].value_counts().sum() == len(df)) # check if total rows of outcome equald total rows

True


In [ ]:
# spot check sales
cols = ["invoice", "stock_code", "description", "quantity", "price", "customer_id"]

df.loc[df["outcome"] == "sales", cols].sample(5, random_state=1) # shows 5 random rows with sales label


,invoice,stock_code,description,quantity,price,customer_id
1064337,581420,20719,WOODLAND CHARLOTTE BAG,10,0.85,16358.0
996052,576617,22076,6 RIBBONS EMPIRE,2,1.25,NaN
837826,564352,22629,SPACEBOY LUNCH BOX,12,1.95,14262.0
752450,556813,22063,CERAMIC BOWL WITH STRAWBERRY DESIGN,1,5.79,NaN
358698,524163,84836,ZINC METAL HEART DECORATION,3,1.25,17524.0


In [ ]:
# spot check cancellations
cols = ["invoice", "stock_code", "description", "quantity", "price", "customer_id"]

df.loc[df["outcome"] == "cancellation", cols].sample(5, random_state=1) # shows 5 random rows with cancel label

,invoice,stock_code,description,quantity,price,customer_id
446624,C531871,21452,TOADSTOOL MONEY BOX,-1,2.95,14105.0
288510,C517465,22061,LARGE CAKE STAND HANGING STRAWBERY,-1,9.95,17865.0
279124,C516565,22269,EGG CUP NATURAL CHICKEN,-8,1.25,12523.0
236966,C512326,85123A,WHITE HANGING HEART T-LIGHT HOLDER,-4,2.55,16001.0
730157,C554693,23203,JUMBO BAG DOILEY PATTERNS,-10,2.08,14292.0


In [13]:
# spot check rejections for bad quantity
cols = ["invoice", "stock_code", "description", "quantity", "price", "customer_id"]

df.loc[df["outcome"] == "rejected: stock adjustement", cols].sample(5, random_state=1) # shows 5 random rows with rej label

,invoice,stock_code,description,quantity,price,customer_id
523460,538061,21801,NaN,-38,0.0,NaN
815151,562279,21391,NaN,-26,0.0,NaN
221114,510907,90092,NaN,-123,0.0,NaN
80319,496734,82096,NaN,-1,0.0,NaN
493958,535635,22962,NaN,-100,0.0,NaN


In [14]:
# spot check rejections for bad price
cols = ["invoice", "stock_code", "description", "quantity", "price", "customer_id"]

df.loc[df["outcome"] == "rejected: bad price", cols].sample(5, random_state=1) # shows 5 random rows with rej label

,invoice,stock_code,description,quantity,price,customer_id
291659,517814,20724,RED SPOTTY CHARLOTTE BAG,2,0.0,NaN
80316,496733,82011E,NaN,1,0.0,NaN
718660,553521,22665,RECIPE BOX BLUE SKETCHBOOK DESIGN,2,0.0,NaN
459201,532832,22950,NaN,16,0.0,NaN
99523,498750,90002A,NaN,7,0.0,NaN
